In [22]:
from utils import get_dataset_lines

# Global Alignment Problem
Find the highest-scoring alignment between two strings using a scoring matrix.

**Code Challenge**: Solve the Global Alignment Problem.

**Input**: A match reward, a mismatch penalty, an indel penalty, and two nucleotide strings.

**Output**: The maximum alignment score of these strings followed by an alignment achieving this maximum score.

**Sample Input**:

```
1 1 2
GAGA
GAT
```

**Sample Output**:

```
-1
GAGA
GA-T
```

In [23]:
def GlobalAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2):
    n = len(s1)
    m = len(s2)
    
    # Initialize score matrix
    score = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Initialize first row and column
    for i in range(1, n + 1):
        score[i][0] = -indel_penalty * i
    for j in range(1, m + 1):
        score[0][j] = -indel_penalty * j
        
    # Fill the score matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score[i-1][j-1] + (match_reward if s1[i-1] == s2[j-1] else -mismatch_penalty)
            delete = score[i-1][j] - indel_penalty
            insert = score[i][j-1] - indel_penalty
            score[i][j] = max(match, delete, insert)
            
    max_score = score[n][m]
    
    # Backtracking
    align1 = ""
    align2 = ""
    i, j = n, m
    
    while i > 0 and j > 0:
        current_score = score[i][j]
        if current_score == score[i-1][j-1] + (match_reward if s1[i-1] == s2[j-1] else -mismatch_penalty):
            align1 += s1[i-1]
            align2 += s2[j-1]
            i -= 1
            j -= 1
        elif current_score == score[i-1][j] - indel_penalty:
            align1 += s1[i-1]
            align2 += "-"
            i -= 1
        else:
            align1 += "-"
            align2 += s2[j-1]
            j -= 1
            
    while i > 0:
        align1 += s1[i-1]
        align2 += "-"
        i -= 1
    while j > 0:
        align1 += "-"
        align2 += s2[j-1]
        j -= 1
        
    return max_score, align1[::-1], align2[::-1]

In [24]:
# Sample Input
match_reward = 1
mismatch_penalty = 1
indel_penalty = 2
s1 = "GAGA"
s2 = "GAT"

# Run the function
score, align1, align2 = GlobalAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2)

# Print Output
print(score)
print(align1)
print(align2)

# Test Assertion
expected_score = -1
expected_align1 = "GAGA"
expected_align2 = "GA-T"

assert score == expected_score
assert align1 == expected_align1
assert align2 == expected_align2
print("Test passed!")

-1
GAGA
GA-T
Test passed!


In [25]:
# Test Dataset
test_dataset_filename = 'dataset_30199_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    match_reward, mismatch_penalty, indel_penalty = map(int, lines[0].split())
    s1 = lines[1].strip()
    s2 = lines[2].strip()
    
    score, align1, align2 = GlobalAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2)
    print(score)
    print(align1)
    print(align2)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

-144
GCTCAATGTTTCAACCGAGGCTAACTCAACTTTTCGGTTTCCGGCATACA-A--A----GTAGCG-CAGTATAAGTAC---AG--TTTCAGAATAAATAGCTACCGGGTCATGCGTGTAGCGGGTGCGG--AAACCGTGCAGCAGCTACTAACCGGCA-GCCAAT-C-CCAATAGTACACAAGTCACGTGGCGTGCGCATGTA-G-GTTCGGTGCCCCCAG-AA-CC-TGAAGCCTTCTATTGCCCTAT-AACT-AA-TAG-C-TTAGTAAGGCCATGGGGTCCAGTAG-CTTTCGTATGTGTACGTTTGATTTCTGCAAG--TGA-CGCATG-GGCTTGTGGCTCCAGCATGTAGAAG-TTTTGCCTCTGGGCCCGCCGTAAATGCCTCCTCAGGATAGCTCCAGATTACGCACTCATACTTTACTCCCGAAATAATCAGGATCAGCGGTCTTAGAGAGTCTAGGAGCTAA-AATGATCGTAATCG--ATAAATATAG-G--TGTTTCATAGCAATACATCACGCGCATTTGTCCCGAAGGGTC-CTCCACG-GGTTTCTTATCGCGACATGATTTGAGGAGTATTTTTATGACGAAGGGTCCCGGCAAGGGTCAGCTATTC-A-GCACCACCGCTTCAACTCG-GCAGGGAAGTGATGGTGAG-A--GCACC---T-C-A-------CTTATTTTAGGG-G-GC--T--AGATCAAGGTCGCCCCACAG-GGCCTGGATG-ACGGCACGCCAAGTTCTTCCCCTAGCCGAGATTTCACGATGCTCTTGTAACTCAAAT-TGA-A--G-AATTC---ATAGA-GGAC-ATCACGGCGCTACTCTGAACGAT-TA-TCAG--A-T-A---TCTACCATACGCGCCTCGAAGAG--TAGCAGA-AAACCTTATGTTCTCATATTGGCTTGCCTCTCTGGTAATTG-CCAACGATCGATACAGAACTTCTCGATTGACGGGTATCCGTTTA-GA
GC

# Local Alignment Problem
Find the highest-scoring local alignment between two strings.

**Code Challenge**: Solve the Local Alignment Problem.

**Input**: Two protein strings written in the single-letter amino acid alphabet.

**Output**: The maximum score of a local alignment of the strings, followed by a local alignment of these strings achieving the maximum score. Use the PAM250 scoring matrix for matches and mismatches as well as the indel penalty $\sigma = 5$.

[PAM250 scoring matrix](PAM250.txt)

**Sample Input**:

```
MEANLY
PENALTY
```

**Sample Output**:

```
15
EANL-Y
ENALTY
```

In [26]:
def read_scoring_matrix(filename):
    scoring_matrix = {}
    with open(filename, 'r') as f:
        lines = f.readlines()
        amino_acids = lines[0].split()
        for line in lines[1:]:
            parts = line.split()
            aa1 = parts[0]
            scores = list(map(int, parts[1:]))
            for i, aa2 in enumerate(amino_acids):
                scoring_matrix[(aa1, aa2)] = scores[i]
    return scoring_matrix

def LocalAlignment(s1, s2, scoring_matrix, indel_penalty):
    n = len(s1)
    m = len(s2)
    
    score = [[0] * (m + 1) for _ in range(n + 1)]
    
    max_score = 0
    max_i, max_j = 0, 0
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score[i-1][j-1] + scoring_matrix[(s1[i-1], s2[j-1])]
            delete = score[i-1][j] - indel_penalty
            insert = score[i][j-1] - indel_penalty
            score[i][j] = max(0, match, delete, insert)
            
            if score[i][j] > max_score:
                max_score = score[i][j]
                max_i, max_j = i, j
                
    # Backtracking
    align1 = ""
    align2 = ""
    i, j = max_i, max_j
    
    while i > 0 and j > 0 and score[i][j] > 0:
        if score[i][j] == score[i-1][j-1] + scoring_matrix[(s1[i-1], s2[j-1])]:
            align1 += s1[i-1]
            align2 += s2[j-1]
            i -= 1
            j -= 1
        elif score[i][j] == score[i-1][j] - indel_penalty:
            align1 += s1[i-1]
            align2 += "-"
            i -= 1
        else:
            align1 += "-"
            align2 += s2[j-1]
            j -= 1
            
    return max_score, align1[::-1], align2[::-1]

In [27]:
# Sample Input
s1 = "MEANLY"
s2 = "PENALTY"
indel_penalty = 5
scoring_matrix = read_scoring_matrix('PAM250.txt')

# Run the function
score, align1, align2 = LocalAlignment(s1, s2, scoring_matrix, indel_penalty)

# Print Output
print(score)
print(align1)
print(align2)

# Test Assertion
expected_score = 15
expected_align1 = "EANL-Y"
expected_align2 = "ENALTY"

assert score == expected_score
assert align1 == expected_align1
assert align2 == expected_align2
print("Test passed!")

15
EANL-Y
ENALTY
Test passed!


In [28]:
# Test Dataset
# Server error: We can not prepare challenge for you right now. Probably, server overloaded. Please try again later.
test_dataset_filename = 'dataset_local_alignment.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    s1 = lines[0].strip()
    s2 = lines[1].strip()
    
    score, align1, align2 = LocalAlignment(s1, s2, scoring_matrix, indel_penalty)
    print(score)
    print(align1)
    print(align2)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

File dataset_local_alignment.txt not found. Please download the dataset and check the filename.


# Edit Distance Problem
Find the edit distance between two strings.

**Input**: Two strings.

**Output**: The edit distance between these strings.

**Code Challenge**: Solve the Edit Distance Problem.

**Sample Input**:

```
GAGA
GAT
```

**Sample Output**:

```
2
```

In [29]:
def EditDistance(s1, s2):
    n = len(s1)
    m = len(s2)
    
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
        
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i-1] == s2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j] + 1,      # deletion
                           dp[i][j-1] + 1,      # insertion
                           dp[i-1][j-1] + cost) # substitution
                           
    return dp[n][m]

In [30]:
# Sample Input
s1 = "GAGA"
s2 = "GAT"

# Run the function
result = EditDistance(s1, s2)

# Print Output
print(result)

# Test Assertion
expected_output = 2
assert result == expected_output
print("Test passed!")

2
Test passed!


In [31]:
# Test Dataset
test_dataset_filename = 'dataset_30200_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    s1 = lines[0].strip()
    s2 = lines[1].strip()
    
    print(EditDistance(s1, s2))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

318


# Fitting Alignment Problem
Find the highest-scoring fitting alignment between two strings.

**Code Challenge**: Solve the Fitting Alignment Problem.

**Input**: Two amino acid strings.

**Output**: A highest-scoring fitting alignment between *v* and *w*. Use the BLOSUM62 scoring table and an indel penalty equal to 1.

Download [BLOSUM62 scoring matrix](BLOSUM62.txt)

**Sample Input**:

```
DISCREPANTLY
PATENT
```

**Sample Output**:

```
20
PA--NT
PATENT
```

In [32]:
def FittingAlignment(s1, s2, scoring_matrix, indel_penalty):
    """
    Fits s1 (shorter string, pattern) into s2 (longer string, text).
    Finds a substring of s2 that has high similarity with all of s1.
    """
    n = len(s1)
    m = len(s2)
    
    score = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Initialize first column (cannot skip prefix of s1 - must match all of s1)
    for i in range(1, n + 1):
        score[i][0] = -indel_penalty * i
    # Initialize first row (can skip prefix of s2 - can start match anywhere in s2)
    for j in range(1, m + 1):
        score[0][j] = 0
        
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score[i-1][j-1] + scoring_matrix[(s1[i-1], s2[j-1])]
            delete = score[i-1][j] - indel_penalty
            insert = score[i][j-1] - indel_penalty
            score[i][j] = max(match, delete, insert)
            
    # Find max score in the last row (can skip suffix of s2 - can end match anywhere in s2)
    # We must reach the last row (n), meaning we have used all of s1.
    max_score = -float('inf')
    max_j = 0
    for j in range(m + 1):
        if score[n][j] > max_score:
            max_score = score[n][j]
            max_j = j
            
    # Backtracking
    align1 = ""
    align2 = ""
    i, j = n, max_j
    
    while i > 0:
        if j > 0 and score[i][j] == score[i-1][j-1] + scoring_matrix[(s1[i-1], s2[j-1])]:
            align1 += s1[i-1]
            align2 += s2[j-1]
            i -= 1
            j -= 1
        elif score[i][j] == score[i-1][j] - indel_penalty:
            align1 += s1[i-1]
            align2 += "-"
            i -= 1
        else:
            align1 += "-"
            align2 += s2[j-1]
            j -= 1
            
    return max_score, align1[::-1], align2[::-1]

In [33]:
# Sample Input
s1 = "DISCREPANTLY"
s2 = "PATENT"
indel_penalty = 1
scoring_matrix = read_scoring_matrix('BLOSUM62.txt')

# Run the function
# We want to fit the shorter sequence (s2) into the longer sequence (s1).
# FittingAlignment(pattern, text) -> (score, aligned_pattern, aligned_text)
score, align_pattern, align_text = FittingAlignment(s2, s1, scoring_matrix, indel_penalty)

# The sample output shows s1 (text) first, then s2 (pattern)
# Because s1 was the first input and s2 was the second input.
align1 = align_text
align2 = align_pattern

# Print Output
print(score)
print(align1)
print(align2)

# Test Assertion
expected_score = 20
expected_align1 = "PA--NT"
expected_align2 = "PATENT"

assert score == expected_score
assert align1 == expected_align1
assert align2 == expected_align2
print("Test passed!")

20
PA--NT
PATENT
Test passed!


In [34]:
# Test Dataset
# Server error: We can not prepare challenge for you right now. Probably, server overloaded. Please try again later.
test_dataset_filename = 'dataset_fitting_alignment.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    s1 = lines[0].strip()
    s2 = lines[1].strip()
    
    # Fit the shorter string into the longer string
    if len(s1) < len(s2):
        # s1 is pattern, s2 is text
        score, align_pattern, align_text = FittingAlignment(s1, s2, scoring_matrix, indel_penalty)
        # Print s1 (pattern) then s2 (text)
        print(score)
        print(align_pattern)
        print(align_text)
    else:
        # s2 is pattern, s1 is text
        score, align_pattern, align_text = FittingAlignment(s2, s1, scoring_matrix, indel_penalty)
        # Print s1 (text) then s2 (pattern)
        print(score)
        print(align_text)
        print(align_pattern)
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

File dataset_fitting_alignment.txt not found. Please download the dataset and check the filename.


# Overlap Alignment Problem
Find the highest-scoring overlap alignment between two strings.

**Code Challenge**: Solve the Overlap Alignment Problem.

**Input**: A match reward, a mismatch penalty, an indel penalty, and two nucleotide strings *v* and *w*.

**Output**: The score of an optimal overlap alignment of *v* and *w*, followed by an alignment of a suffix *v'* of *v* and a prefix *w'* of *w* achieving this maximum score.

**Sample Input**:

```
1 1 2
GAGA
GAT
```

**Sample Output**:

```
2
GA
GA
```

In [35]:
def OverlapAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2):
    n = len(s1)
    m = len(s2)
    
    score = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Initialize first column (can skip prefix of s1)
    for i in range(1, n + 1):
        score[i][0] = 0
    # Initialize first row (cannot skip prefix of s2)
    for j in range(1, m + 1):
        score[0][j] = -indel_penalty * j
        
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score[i-1][j-1] + (match_reward if s1[i-1] == s2[j-1] else -mismatch_penalty)
            delete = score[i-1][j] - indel_penalty
            insert = score[i][j-1] - indel_penalty
            score[i][j] = max(match, delete, insert)
            
    # Find max score in the last row (must use all of s1, can use prefix of s2)
    max_score = -float('inf')
    max_j = 0
    for j in range(m + 1):
        if score[n][j] >= max_score: # Use >= to prefer longer alignment? Or just >?
            max_score = score[n][j]
            max_j = j
            
    # Backtracking
    align1 = ""
    align2 = ""
    i, j = n, max_j
    
    while i > 0 and j > 0:
        current_score = score[i][j]
        if current_score == score[i-1][j-1] + (match_reward if s1[i-1] == s2[j-1] else -mismatch_penalty):
            align1 += s1[i-1]
            align2 += s2[j-1]
            i -= 1
            j -= 1
        elif current_score == score[i-1][j] - indel_penalty:
            align1 += s1[i-1]
            align2 += "-"
            i -= 1
        else:
            align1 += "-"
            align2 += s2[j-1]
            j -= 1
            
    return max_score, align1[::-1], align2[::-1]

In [36]:
# Sample Input
match_reward = 1
mismatch_penalty = 1
indel_penalty = 2
s1 = "GAGA"
s2 = "GAT"

# Run the function
score, align1, align2 = OverlapAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2)

# Print Output
print(score)
print(align1)
print(align2)

# Test Assertion
expected_score = 2
expected_align1 = "GA"
expected_align2 = "GA"

assert score == expected_score
assert align1 == expected_align1
assert align2 == expected_align2
print("Test passed!")

2
GA
GA
Test passed!


In [37]:
# Test Dataset
test_dataset_filename = 'dataset_30200_7.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    match_reward, mismatch_penalty, indel_penalty = map(int, lines[0].split())
    s1 = lines[1].strip()
    s2 = lines[2].strip()
    
    score, align1, align2 = OverlapAlignment(match_reward, mismatch_penalty, indel_penalty, s1, s2)
    print(score)
    print(align1)
    print(align2)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

187
CACCGAGGAACAATCAACTTCCGCTTGGTTCCCAGGTCCACTATGGGCATTTTGTCGTCCAGGCTGAA--CCT--GGCGGCTGGATGATGGGAC-CGCTGG-----GGGGTGGA-A-CT-AGAGAGTGAAGGTAGGAC----CAA-TCGGGTTAGGGAGGGT-G--ACTGGTGAGTTTGGTGC--CCGATCAAGA--CTATCAC-TCAGCGCAAT----ACCCAGCCCGGGTTCTTTCAACGGACGGGACGCTA-GTGCCCATGACCA-TCGGCGGGAATCTTGGCAACGTTCGCTACCATTTGACAG-ACG-GGGCTTATCAGAGACAGTACGGTTCCTATTTAGGTACGGGGT-CA--T-GATCACGACCTAGCACTCCGATCCTGCA---CCACACCGTCGTG---CACGGTACTCTTTCG--AGCGAC--CGG--A--AGCGCTTACCCGGGTTTCTTGTTTG-GTTGATTTGCAGACCCACTGTCGCAAATCTCCAGAA-CTGGG--TTTTTTC-TGT-GTTTGCC-C-CATTGTTCTCTTGCCGTCGCGGGCT-ATCAGAGCCTAACGATGGACTCGCAGAACCGACCCGAGATAAACGCTTAATGTTGCTAGCAGGCTACGACCATAGGCGGTGGTACATACCCTTGGTAGCCCGCCGCGTTCAATTTCCGTATACGCCCGCTCCAGCTAGGATATTGACGCGAA--GC-CGGGTGAGCAGACTCAGCTCGAATCTATCAA---C---C-CGCGGA-C--G-CATACTAGC--A--GCAAAGC-CTC--GAT-CCCAAAG-CGCGAG-C--C-ATC-TTTCCGTCCTGA--TGATTTTTACTGCTA-CAGGCGAGCAGGCGTTGCAGAGATACGCCTCCATGCAAGTGGTG--GG--GC--CGGAAGTAGTGCGATACGTTTCACCAGCGGTCTATTCGAAAAAATCTTCCACTCTGGCTTTGTGTGATACATCTATACAGAA
CACCG